# 结构化输出

以固定字段约定组织输出，再用程序验证格式与业务约束。

按章节从上到下运行。说明、代码、输出应一起阅读。所有模型调用都访问真实服务并可能计费，重新运行也会产生请求。示例制度和数字仅用于课堂练习。


In [2]:
import json
from pathlib import Path
from course_api import pretty, answer, text, chat, vision, image_part, config_status
from IPython.display import display, Image, Markdown
pretty(config_status())


{
  "Python配置": "请同时检查 sys.executable",
  "对话模型": "deepseek-flash",
  "对话地址": "https://api.deepseek.com",
  "对话密钥已配置": true
}


## 1. 字段约定

Pydantic 约定类型、长度和可选值。缺失字段与类型错误应在进入业务逻辑前发现。


In [3]:
from pydantic import BaseModel, Field, ConfigDict, ValidationError
class StudyCard(BaseModel):
    model_config = ConfigDict(extra='forbid')
    topic: str = Field(min_length=1, max_length=80)
    points: list[str] = Field(min_length=1, max_length=5)
    uncertainty: str
pretty(StudyCard.model_json_schema())


{
  "additionalProperties": false,
  "properties": {
    "topic": {
      "maxLength": 80,
      "minLength": 1,
      "title": "Topic",
      "type": "string"
    },
    "points": {
      "items": {
        "type": "string"
      },
      "maxItems": 5,
      "minItems": 1,
      "title": "Points",
      "type": "array"
    },
    "uncertainty": {
      "title": "Uncertainty",
      "type": "string"
    }
  },
  "required": [
    "topic",
    "points",
    "uncertainty"
  ],
  "title": "StudyCard",
  "type": "object"
}


### 运行后观察

schema 描述程序需要的字段。uncertainty 允许空字符串，但要求模型明确返回该字段。


## 2. JSON模式

JSON模式要求请求中说明JSON。它帮助输出有效JSON，字段类型与含义仍由本地校验。


In [4]:
prompt = ('只输出JSON对象，字段为topic字符串、points字符串列表、uncertainty字符串。'
          '将下面内容整理成学习卡：Python列表可变，元组不可变。不要添加没有给出的性能结论。')
raw = text([{'role':'user','content':prompt}], json_mode=True)
import json
pretty(json.loads(raw))


{
  "topic": "Python 列表与元组的可变性",
  "points": [
    "Python 列表可变",
    "元组不可变"
  ],
  "uncertainty": "未提供性能相关结论"
}


### 运行后观察

观察双引号、数组与对象层级。漂亮的JSON也可能填入错误知识。


## 3. 校验与错误

先解析JSON，再验证schema。下面的错误样例是本地数据校验练习，不产生API调用。


In [5]:
card = StudyCard.model_validate_json(raw)
pretty(card.model_dump())
try:
    StudyCard.model_validate({'topic':'列表', 'points':'这本应是列表', 'uncertainty':''})
except ValidationError as exc:
    pretty(exc.errors(include_url=False))


{
  "topic": "Python 列表与元组的可变性",
  "points": [
    "Python 列表可变",
    "元组不可变"
  ],
  "uncertainty": "未提供性能相关结论"
}
[
  {
    "type": "list_type",
    "loc": [
      "points"
    ],
    "msg": "Input should be a valid list",
    "input": "这本应是列表"
  }
]


### 运行后观察

错误位置points指向不符合约定的字段。JSON合法性、字段合法性、事实正确性是不同检查。


## 4. 有界重试

将具体校验错误反馈一次。达到上限后抛出错误，不返回伪造成功结果。


In [6]:
def validated_card(source, attempts=2):
    schema = json.dumps(StudyCard.model_json_schema(), ensure_ascii=False)
    messages = [{'role':'system','content':'输出符合schema的JSON对象。只依据原文，模糊处写入uncertainty。schema:'+schema},
                {'role':'user','content':source}]
    for attempt in range(attempts):
        result = text(messages, json_mode=True)
        try:
            return StudyCard.model_validate_json(result)
        except ValidationError as exc:
            if attempt+1 == attempts:
                raise
            messages += [{'role':'assistant','content':result},
                         {'role':'user','content':'请修正JSON字段错误：'+str(exc)}]
card = validated_card('列表支持追加元素，元组创建后不能修改元素。')
pretty(card.model_dump())


{
  "topic": "列表与元组的可变性差异",
  "points": [
    "列表支持追加元素",
    "元组创建后不能修改元素"
  ],
  "uncertainty": "未说明追加元素的具体方法，也未说明元组元素不可修改的具体含义（如是否完全不可变）"
}


### 运行后观察

最多两次真实请求。格式修复成功后仍需核对原文。


## 5. 通知字段与业务约束

类型校验之外还需要关系约束。本例使用date表示日期，None表示未知。deadline不能晚于活动日期。证据原句以字典保存，格式通过后仍需核对原图。


In [7]:
from datetime import date
from pydantic import model_validator
class NoticeRecord(BaseModel):
    model_config = ConfigDict(extra='forbid')
    name: str = Field(min_length=1)
    location: str | None
    event_date: date | None
    capacity: int | None = Field(ge=1, le=1000)
    deadline: date | None
    evidence: dict[str, str]
    @model_validator(mode='after')
    def check_dates(self):
        if self.deadline and self.event_date and self.deadline > self.event_date:
            raise ValueError('报名截止不能晚于活动日期')
        return self
pretty(NoticeRecord.model_json_schema())


{
  "additionalProperties": false,
  "properties": {
    "name": {
      "minLength": 1,
      "title": "Name",
      "type": "string"
    },
    "location": {
      "anyOf": [
        {
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "title": "Location"
    },
    "event_date": {
      "anyOf": [
        {
          "format": "date",
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "title": "Event Date"
    },
    "capacity": {
      "anyOf": [
        {
          "maximum": 1000,
          "minimum": 1,
          "type": "integer"
        },
        {
          "type": "null"
        }
      ],
      "title": "Capacity"
    },
    "deadline": {
      "anyOf": [
        {
          "format": "date",
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "title": "Deadline"
    },
    "evidence": {
      "additionalProperties": {
        "type": 

### 运行后观察

有字段不代表有值，None与空字符串含义不同。event_date是活动日期，deadline是报名截止，两者不能互相代填。


## 6. JSON合法但业务错误

下面是刻意构造的本地校验样本，用于观察程序错误，与真实API回复无关。第一条人数错误，第二条日期关系错误。


In [8]:
bad_cases = [
    {'name':'AI学习工作坊','location':'A201','event_date':'2026-11-02','capacity':-2,
     'deadline':None,'evidence':{}},
    {'name':'AI学习工作坊','location':'A201','event_date':'2026-11-02','capacity':20,
     'deadline':'2026-11-03','evidence':{}}
]
for i, item in enumerate(bad_cases, 1):
    try:
        NoticeRecord.model_validate(item)
    except ValidationError as exc:
        pretty({'case':i,'errors':exc.errors(include_url=False,include_context=False)})


{
  "case": 1,
  "errors": [
    {
      "type": "greater_than_equal",
      "loc": [
        "capacity"
      ],
      "msg": "Input should be greater than or equal to 1",
      "input": -2
    }
  ]
}
{
  "case": 2,
  "errors": [
    {
      "type": "value_error",
      "loc": [],
      "msg": "Value error, 报名截止不能晚于活动日期",
      "input": {
        "name": "AI学习工作坊",
        "location": "A201",
        "event_date": "2026-11-02",
        "capacity": 20,
        "deadline": "2026-11-03",
        "evidence": {}
      }
    }
  ]
}


### 运行后观察

JSON解析可以通过，负数人数和不合理日期仍应被拒绝。修复数据必须回到原文，不能随意把负数改成某个正数。


## 7. 真实提取与证据核对

读取同目录通知原文，向真实服务请求符合schema的JSON。此处专门检查结构化阶段，图像识别的影响已经在01单独观察。


In [9]:
source = Path('data/通知原文.txt').read_text(encoding='utf-8')
schema = json.dumps(NoticeRecord.model_json_schema(), ensure_ascii=False)
notice_raw = text([
    {'role':'system','content':'只根据原文合并通知，输出符合schema的JSON。待定或未给出的日期、数字填null。evidence按字段保存连续原句。schema:'+schema},
    {'role':'user','content':source}
], json_mode=True)
notice = NoticeRecord.model_validate_json(notice_raw)
pretty(notice.model_dump(mode='json'))
quote_checks = {field: quote in source for field,quote in notice.evidence.items()}
pretty({'quotes_are_substrings':quote_checks})


{
  "name": "AI学习工作坊招募通知",
  "location": "B305教室",
  "event_date": "2026-11-02",
  "capacity": 30,
  "deadline": null,
  "evidence": {
    "name": "AI学习工作坊招募通知V1。",
    "location": "地点A201实验室。补充说明V2：地点调整为B305教室，名额调整为30人。",
    "event_date": "活动日期2026年11月02日。活动日期沿用原通知，报名截止仍待定。",
    "capacity": "名额20人。补充说明V2：地点调整为B305教室，名额调整为30人。",
    "deadline": "报名截止待定。活动日期沿用原通知，报名截止仍待定。"
  }
}
{
  "quotes_are_substrings": {
    "name": true,
    "location": false,
    "event_date": false,
    "capacity": false,
    "deadline": false
  }
}


### 运行后观察

连续原句存在只是初步证据检查：错误字段也可能引用一段真实但无关的文字。还需人工确认地点为B305、名额30、活动日期沿用V1、截止未知。模型若用省略号或改写原句，子串检查会失败，应人工核对并改善证据约定。


## 8. 错误分类与修复策略

不要把所有失败都放进同一个重试循环。语法/字段错误可以尝试一次有界修复；资料未写明的信息需要保留未知；图片看不清应改善输入。


In [10]:
def classify_notice_error(record):
    checks = {'capacity_positive':record.capacity is None or record.capacity > 0,
              'deadline_known':record.deadline is not None,
              'has_evidence':bool(record.evidence)}
    return checks
pretty(classify_notice_error(notice))


{
  "capacity_positive": true,
  "deadline_known": false,
  "has_evidence": true
}


### 运行后观察

deadline_known=False不是程序失败，它可能准确反映“待定”。区分没有资料与不符合规则，才能决定追问还是修复。
